# 6- RESIDUALS DISTRIBUTION FIGURES

In [3]:
import numpy as np
import pandas as pd
import joblib
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

# ==========================================================
# 1. Load Data
# ==========================================================
data = pd.read_csv("new_csv1.csv", sep=";")

X = data.drop("Calculated Temperature , k", axis=1)
y = data["Calculated Temperature , k"]

# ==========================================================
# 2. Train-Test Split
# ==========================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ==========================================================
# 3. Scaling
# ==========================================================
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ==========================================================
# 4. Load Trained Model
# ==========================================================
model = joblib.load("deep_ann_model.pkl")

# Predict Test Set
y_pred = model.predict(X_test_scaled)

# Create folder for figures
os.makedirs("model_plots", exist_ok=True)

# ==========================================================
# 5. Scatter Plot: True vs Predicted
# ==========================================================
plt.figure(figsize=(7,6))
sns.scatterplot(x=y_test, y=y_pred, s=60)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--', lw=2)
plt.xlabel("True Temperature (K)")
plt.ylabel("Predicted Temperature (K)")
plt.title("True vs Predicted (deep_ann_model)")
plt.grid(True)
plt.savefig("model_plots/true_vs_pred.png", dpi=300)
plt.close()

# ==========================================================
# 6. Residual Plot (Color Mapped)
# ==========================================================
residuals = y_pred - y_test

plt.figure(figsize=(7,6))
sns.scatterplot(
    x=y_pred,
    y=residuals,
    hue=residuals,
    palette="coolwarm",
    s=60
)
plt.axhline(0, color='black', linestyle='--')
plt.xlabel("Predicted Temperature (K)")
plt.ylabel("Residual (Pred - True)")
plt.title("Residuals vs Predicted (Color Mapped)")
plt.grid(True)
plt.savefig("model_plots/residuals_vs_pred_colormap.png", dpi=300)
plt.close()

# ==========================================================
# 7. Distribution of Residuals
# ==========================================================
plt.figure(figsize=(7,6))
sns.histplot(residuals, kde=True, color='blue')
plt.xlabel("Residual (Predicted - True)")
plt.ylabel("Count")
plt.title("Residual Distribution (deep_ann_model)")
plt.grid(True)
plt.savefig("model_plots/residual_distribution.png", dpi=300)
plt.close()

# ==========================================================
# 8. Heatmap of Absolute Residuals
# ==========================================================
plt.figure(figsize=(7,6))
plt.scatter(
    range(len(residuals)),
    y_test,
    c=np.abs(residuals),
    cmap='viridis',
    s=60
)
cbar = plt.colorbar()
cbar.set_label("Absolute Residual (K)")
plt.xlabel("Sample Index")
plt.ylabel("True Temperature (K)")
plt.title("Absolute Residual Color Map")
plt.grid(True)
plt.savefig("model_plots/abs_residual_color_map.png", dpi=300)
plt.close()

# ==========================================================
# 9. Error Metrics
# ==========================================================
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("===========================================")
print(" deep_ann_model Performance (TEST DATA) ")
print("===========================================")
print(f"MSE  : {mse:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R²   : {r2:.4f}")
print("Figures saved in: model_plots/")


 deep_ann_model Performance (TEST DATA) 
MSE  : 9.2417
RMSE : 3.0400
R²   : 0.9371
Figures saved in: model_plots/


In [10]:
import numpy as np
import pandas as pd
import joblib
import os
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import seaborn as sns

# ==========================================================
# Load Data
# ==========================================================
data = pd.read_csv("new_csv1.csv", sep=";")

X = data.drop("Calculated Temperature , k", axis=1)
y = data["Calculated Temperature , k"]

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Load model
model = joblib.load("deep_ann_model.pkl")

y_pred = model.predict(X_test_scaled)

# Create folder
os.makedirs("paper_figures", exist_ok=True)

# ==========================================================
# 1) True vs Predicted (with density colormap)
# ==========================================================
plt.figure(figsize=(8, 7))
plt.scatter(y_test, y_pred, c=np.abs(y_pred - y_test),
            cmap="viridis", s=60, edgecolor="k")

plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()],
         'r--', linewidth=2)

cbar = plt.colorbar()
cbar.set_label("Absolute Error (K)", fontsize=12)

plt.xlabel("True Temperature (K)", fontsize=14)
plt.ylabel("Predicted Temperature (K)", fontsize=14)
plt.title("True vs Predicted Temperature (deep_ann_model)", fontsize=16)
plt.grid(True, linestyle="--", alpha=0.5)

plt.savefig("paper_figures/true_vs_pred_scientific.png", dpi=400, bbox_inches="tight")
plt.close()

# ==========================================================
# 2) Residuals vs Predicted (color mapped)
# ==========================================================
residuals = y_pred - y_test

plt.figure(figsize=(8, 7))
plt.scatter(y_pred, residuals, c=residuals,
            cmap="coolwarm", s=60, edgecolor="k")

plt.axhline(0, color="black", linestyle="--", linewidth=2)

cbar = plt.colorbar()
cbar.set_label("Residual (K)", fontsize=12)

plt.xlabel("Predicted Temperature (K)", fontsize=14)
plt.ylabel("Residual (Pred - True)", fontsize=14)
plt.title("Residuals vs Predicted (deep_ann_model)", fontsize=16)
plt.grid(True, linestyle="--", alpha=0.5)

plt.savefig("paper_figures/residuals_vs_pred_scientific.png", dpi=400, bbox_inches="tight")
plt.close()

# ==========================================================
# Print Metrics
# ==========================================================
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("===========================================")
print(" deep_ann_model Performance (TEST DATA) ")
print("===========================================")
print(f"MSE  : {mse:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R²   : {r2:.4f}")
print("Figures saved in: paper_figures/")


 deep_ann_model Performance (TEST DATA) 
MSE  : 9.2417
RMSE : 3.0400
R²   : 0.9371
Figures saved in: paper_figures/


In [15]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
from xgboost import XGBRegressor

# ----------------------------------------------------
# 1. LOAD DATA
# ----------------------------------------------------
data = pd.read_csv("output.csv")

# Features and Target
X = data.drop("Temperature (K)", axis=1)
y = data["Temperature (K)"]

# ----------------------------------------------------
# 2. TRAIN–TEST SPLIT
# ----------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ----------------------------------------------------
# 3. DEFINE XGBOOST MODEL
# ----------------------------------------------------
xgb_model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

# ----------------------------------------------------
# 4. TRAIN MODEL
# ----------------------------------------------------
xgb_model.fit(X_train, y_train)

# ----------------------------------------------------
# 5. PREDICT & EVALUATE
# ----------------------------------------------------
y_pred_train = xgb_model.predict(X_train)
y_pred_test = xgb_model.predict(X_test)

# R2 scores
r2_train = r2_score(y_train, y_pred_train)
r2_test = r2_score(y_test, y_pred_test)

# MSE scores
mse_train = mean_squared_error(y_train, y_pred_train)
mse_test = mean_squared_error(y_test, y_pred_test)

print("\n================ XGBOOST RESULTS ================")
print(f"Train R²: {r2_train:.5f}")
print(f"Test  R²: {r2_test:.5f}")
print(f"Train MSE: {mse_train:.5f}")
print(f"Test  MSE: {mse_test:.5f}")
print("=================================================\n")



================ XGBOOST RESULTS ================
Train R²: 0.99996
Test  R²: 0.99909
Train MSE: 0.00892
Test  MSE: 0.19273



In [16]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
from xgboost import XGBRegressor

# ----------------------------------------------------
# 1. LOAD DATA
# ----------------------------------------------------
data = pd.read_csv("output.csv")

X = data.drop("Temperature (K)", axis=1)
y = data["Temperature (K)"]

# ----------------------------------------------------
# 2. TRAIN–TEST SPLIT (same as your training)
# ----------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ----------------------------------------------------
# 3. LOAD THE TRAINED MODEL
# ----------------------------------------------------
loaded_xgb = XGBRegressor()
loaded_xgb.load_model("xgb_hydrate_model.json")

# ----------------------------------------------------
# 4. PREDICT
# ----------------------------------------------------
y_pred = loaded_xgb.predict(X_test)

# ----------------------------------------------------
# 5. METRICS
# ----------------------------------------------------
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("==============================================")
print("  XGBoost Hydrate Model — Evaluation Metrics")
print("==============================================")
print(f"Test MSE : {mse:.4f}")
print(f"Test R²  : {r2:.4f}")


  XGBoost Hydrate Model — Evaluation Metrics
Test MSE : 0.1714
Test R²  : 0.9992


In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
from xgboost import XGBRegressor
import os

# =====================================================
# 1. LOAD DATA
# =====================================================
data = pd.read_csv("output.csv")

X = data.drop("Temperature (K)", axis=1)
y = data["Temperature (K)"]

# =====================================================
# 2. TRAIN–TEST SPLIT  (exactly same settings)
# =====================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# =====================================================
# 3. LOAD THE TRAINED XGBOOST MODEL
# =====================================================
loaded_xgb = XGBRegressor()
loaded_xgb.load_model("xgb_hydrate_model.json")

# =====================================================
# 4. PREDICT
# =====================================================
y_pred = loaded_xgb.predict(X_test)

# =====================================================
# 5. METRICS
# =====================================================
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("==============================================")
print("  XGBoost Hydrate Model — Evaluation Metrics")
print("==============================================")
print(f"Test MSE : {mse:.4f}")
print(f"Test R²  : {r2:.4f}")

# Create folder for figures
os.makedirs("paper_figures", exist_ok=True)

# =====================================================
# 6. FIGURE 1 — TRUE vs PREDICTED (colored by error)
# =====================================================
abs_error = np.abs(y_pred - y_test)

plt.figure(figsize=(8, 7))
sc = plt.scatter(y_test, y_pred, c=abs_error,
                 cmap="viridis", s=60, edgecolor="k")

plt.plot([y_test.min(), y_test.max()],
         [y_test.min(), y_test.max()],
         "r--", linewidth=2)

plt.xlabel("True Temperature (K)", fontsize=14)
plt.ylabel("Predicted Temperature (K)", fontsize=14)
plt.title("True vs Predicted Hydrate Temperature (XGBoost)", fontsize=16)

cbar = plt.colorbar(sc)
cbar.set_label("Absolute Error (K)", fontsize=12)

plt.grid(True, linestyle="--", alpha=0.4)
plt.savefig("paper_figures/true_vs_pred_xgb.png", dpi=400, bbox_inches="tight")
plt.close()

# =====================================================
# 7. FIGURE 2 — RESIDUALS vs PREDICTED
# =====================================================
residuals = y_pred - y_test

plt.figure(figsize=(8, 7))
sc2 = plt.scatter(y_pred, residuals, c=residuals,
                  cmap="coolwarm", s=60, edgecolor="k")

plt.axhline(0, color="black", linestyle="--")

plt.xlabel("Predicted Temperature (K)", fontsize=14)
plt.ylabel("Residual (Pred - True)", fontsize=14)
plt.title("Residuals vs Predicted (XGBoost Hydrate Model)", fontsize=16)

cbar2 = plt.colorbar(sc2)
cbar2.set_label("Residual Value (K)", fontsize=12)

plt.grid(True, linestyle="--", alpha=0.4)
plt.savefig("paper_figures/residuals_vs_pred_xgb.png", dpi=400, bbox_inches="tight")
plt.close()

print("\nFigures saved to folder: paper_figures/")


  XGBoost Hydrate Model — Evaluation Metrics
Test MSE : 0.1714
Test R²  : 0.9992

Figures saved to folder: paper_figures/


In [20]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
from xgboost import XGBRegressor
import matplotlib.pyplot as plt

# ----------------------------------------------------
# 1. LOAD DATA
# ----------------------------------------------------
data = pd.read_csv("output.csv")

X = data.drop("Temperature (K)", axis=1)
y = data["Temperature (K)"]

# ----------------------------------------------------
# 2. TRAIN–TEST SPLIT
# ----------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ----------------------------------------------------
# 3. LOAD TRAINED XGBOOST MODEL
# ----------------------------------------------------
model = XGBRegressor()
model.load_model("xgb_hydrate_model.json")

# ----------------------------------------------------
# 4. PREDICT
# ----------------------------------------------------
y_pred = model.predict(X_test)

# ----------------------------------------------------
# 5. CREATE NEW FOLDER
# ----------------------------------------------------
save_folder = "extra_visualizations"
os.makedirs(save_folder, exist_ok=True)

# ----------------------------------------------------
# 6. FIGURE: Prediction Error Distribution Histogram
# ----------------------------------------------------
errors = y_pred - y_test

plt.figure(figsize=(8,6))
plt.hist(errors, bins=30, edgecolor='black')
plt.xlabel("Prediction Error (K)")
plt.ylabel("Frequency")
plt.title("Distribution of Prediction Errors (XGBoost Hydrate Model)")
plt.grid(True, linestyle="--", alpha=0.5)

# SAVE FIGURE
save_path = os.path.join(save_folder, "error_distribution_histogram.png")
plt.savefig(save_path, dpi=400, bbox_inches="tight")
plt.close()

print("Image saved at:", save_path)


Image saved at: extra_visualizations\error_distribution_histogram.png
